In [ ]:
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Math et visualisation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm.notebook import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# Utilitaires
import yaml
import json
from datetime import datetime

In [ ]:
# Montage Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Configure paths according to your project structure
project_path = '/content/drive/MyDrive/driver-distraction-detection'
config_dir = Path(project_path) / 'config'
data_dir = Path(project_path) / 'data'
images_dir = data_dir / 'raw'
annotations_dir = data_dir / 'annotations'
src_dir = Path(project_path) / 'src'

# Add src to system path
sys.path.append(str(src_dir))

In [ ]:
try:
    import preprocess
    print("✓ Preprocess module imported successfully")
except ImportError as e:
    print(f"Error importing preprocess: {e}")
    sys.exit(1)

# Verify directories exist
print("="*70)
print("PROJECT STRUCTURE VERIFICATION")
print("="*70)
print(f"✓ Project root: {Path(project_path).exists()}")
print(f"✓ Config directory: {config_dir.exists()}")
print(f"✓ Images directory: {images_dir.exists()}")
print(f"✓ Annotations directory: {annotations_dir.exists()}")
print(f"✓ Source directory: {src_dir.exists()}")

✓ Preprocess module imported successfully
PROJECT STRUCTURE VERIFICATION
✓ Project root: True
✓ Config directory: True
✓ Images directory: True
✓ Annotations directory: True
✓ Source directory: True


In [ ]:
# Check class folders in raw directory
if images_dir.exists():
    class_folders = sorted([f for f in os.listdir(images_dir)
                          if os.path.isdir(images_dir / f) and f.startswith('c')])
    print(f"\nFound {len(class_folders)} class folders in raw directory:")
    for folder in class_folders:
        folder_path = images_dir / folder
        image_count = len([f for f in os.listdir(folder_path)
                         if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f"  - {folder}: {image_count} images")


Found 10 class folders in raw directory:
  - c0_safe_driving: 2489 images
  - c1_texting_right: 2267 images
  - c2_talking_phone_right: 2317 images
  - c3_texting_left: 2346 images
  - c4_talking_phone_left: 2326 images
  - c5_operating_radio: 2312 images
  - c6_drinking: 2325 images
  - c7_reaching_behind: 2002 images
  - c8_hair_makeup: 1911 images
  - c9_talking_passenger: 2129 images


In [ ]:
# Load configuration files
try:
    # Training configuration
    with open(config_dir / 'training.yaml', 'r') as f:
        training_config = yaml.safe_load(f)

    # Model configuration
    with open(config_dir / 'model.yaml', 'r') as f:
        model_config = yaml.safe_load(f)

    # Class mapping
    with open(config_dir / 'class_mapping.yaml', 'r') as f:
        class_config = yaml.safe_load(f)

    print("✓ All configurations loaded successfully")

except Exception as e:
    print(f"Error loading configurations: {e}")
    sys.exit(1)

# Display configuration summary
print("\nTraining Configuration:")
print(f"  - Batch size: {training_config['dataloader']['batch_size']}")
print(f"  - Epochs: {training_config['training']['epochs']}")
print(f"  - Learning rate: {training_config['optimizer']['lr']}")
print(f"  - Early stopping patience: {training_config['training']['early_stopping_patience']}")

print("\nAvailable Models:")
for model_name, model_info in model_config['models'].items():
    print(f"  - {model_name}: input_size={model_info['input_size']}, "
          f"pretrained={model_info['pretrained']}")

print("\nClass Mapping:")
for class_id, class_info in sorted(class_config['classes'].items()):
    print(f"  - Class {class_id}: {class_info['name']}")

✓ All configurations loaded successfully

Training Configuration:
  - Batch size: 64
  - Epochs: 25
  - Learning rate: 0.0003
  - Early stopping patience: 7

Available Models:
  - resnet50: input_size=(224,224), pretrained=True
  - efficientnet_b3: input_size=(300,300), pretrained=True

Class Mapping:
  - Class 0: safe_driving
  - Class 1: texting_right
  - Class 2: talking_phone_right
  - Class 3: texting_left
  - Class 4: talking_phone_left
  - Class 5: operating_radio
  - Class 6: drinking
  - Class 7: reaching_behind
  - Class 8: hair_makeup
  - Class 9: talking_passenger


In [ ]:
# ============================================================================
# 3. DATA PATHS AND VERIFICATION
# ============================================================================

print("\n" + "="*70)
print("DATA VERIFICATION")
print("="*70)

# CSV file paths
train_csv = annotations_dir / 'train_labels.csv'
val_csv = annotations_dir / 'val_labels.csv'
test_csv = annotations_dir / 'test_labels.csv'

# Verify CSV files exist
csv_files = {
    'Train': train_csv,
    'Validation': val_csv,
    'Test': test_csv
}

for split_name, csv_path in csv_files.items():
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        print(f"✓ {split_name}: {len(df)} samples ({csv_path.name})")

        # Check sample from CSV
        if len(df) > 0:
            sample = df.iloc[0]
            sample_image_path = images_dir / sample['image_path']
            if sample_image_path.exists():
                print(f"    Sample: {sample['image_path']} → Label: {sample['label']}")
            else:
                print(f"      Warning: Sample image not found: {sample_image_path}")
    else:
        print(f"✗ {split_name}: File not found ({csv_path.name})")


DATA VERIFICATION
✓ Train: 17939 samples (train_labels.csv)
    Sample: c5_operating_radio/img_18541.jpg → Label: c5_operating_radio
✓ Validation: 2242 samples (val_labels.csv)
    Sample: c3_texting_left/img_57710.jpg → Label: c3_texting_left
✓ Test: 2243 samples (test_labels.csv)
    Sample: c4_talking_phone_left/img_43001.jpg → Label: c4_talking_phone_left


In [ ]:
print("\n" + "="*70)
print("CREATING DATALOADERS")
print("="*70)

# Get models to train
MODELS_TO_TRAIN = list(model_config['models'].keys())
print(f"Models to train: {MODELS_TO_TRAIN}")

# Use the first model's input size as default
first_model = MODELS_TO_TRAIN[0]
# Convert the string representation of the tuple to an actual tuple
input_size_str = model_config['models'][first_model]['input_size']
DEFAULT_INPUT_SIZE = tuple(map(int, input_size_str.strip('()').split(',')))
print(f"Using input size: {DEFAULT_INPUT_SIZE}")

# Create transforms
train_transform = preprocess.get_train_transforms(DEFAULT_INPUT_SIZE)
val_transform = preprocess.get_val_transforms(DEFAULT_INPUT_SIZE)
test_transform = preprocess.get_test_transforms(DEFAULT_INPUT_SIZE)

# Dataloader parameters from config
batch_size = training_config['dataloader']['batch_size']
num_workers = training_config['dataloader']['num_workers']

print(f"\nDataloader parameters:")
print(f"  - Batch size: {batch_size}")
print(f"  - Number of workers: {num_workers}")

# Create dataloaders
try:
    train_loader = preprocess.create_dataloader(
        csv_file=str(train_csv),
        images_dir=str(images_dir),
        transform=train_transform,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers
    )

    val_loader = preprocess.create_dataloader(
        csv_file=str(val_csv),
        images_dir=str(images_dir),
        transform=val_transform,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    test_loader = preprocess.create_dataloader(
        csv_file=str(test_csv),
        images_dir=str(images_dir),
        transform=test_transform,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    print("\n✓ Dataloaders created successfully")

    # Calculate and display statistics
    total_samples = (
        len(train_loader.dataset) +
        len(val_loader.dataset) +
        len(test_loader.dataset)
    )

    print(f"\nDataset Statistics:")
    print(f"  - Training samples: {len(train_loader.dataset):,} "
          f"({len(train_loader.dataset)/total_samples*100:.1f}%)")
    print(f"  - Validation samples: {len(val_loader.dataset):,} "
          f"({len(val_loader.dataset)/total_samples*100:.1f}%)")
    print(f"  - Test samples: {len(test_loader.dataset):,} "
          f"({len(test_loader.dataset)/total_samples*100:.1f}%)")
    print(f"  - Total samples: {total_samples:,}")

    # Display batch information
    sample_batch = next(iter(train_loader))
    images, labels = sample_batch
    print(f"\nSample batch shape:")
    print(f"  - Images: {images.shape} (batch_size × channels × height × width)")
    print(f"  - Labels: {labels.shape} (batch_size)")

except Exception as e:
    print(f"Error creating dataloaders: {e}")
    import traceback
    traceback.print_exc()
    sys.exit(1)


CREATING DATALOADERS
Models to train: ['resnet50', 'efficientnet_b3']
Using input size: (224, 224)

Dataloader parameters:
  - Batch size: 64
  - Number of workers: 10

✓ Dataloaders created successfully

Dataset Statistics:
  - Training samples: 17,939 (80.0%)
  - Validation samples: 2,242 (10.0%)
  - Test samples: 2,243 (10.0%)
  - Total samples: 22,424

Sample batch shape:
  - Images: torch.Size([64, 3, 224, 224]) (batch_size × channels × height × width)
  - Labels: torch.Size([64]) (batch_size)


In [ ]:
TEST_MODE = False  # Set to False for full training

if TEST_MODE:
    print("\n" + "="*70)
    print("TEST MODE ACTIVATED - Using reduced dataset")
    print("="*70)

    from torch.utils.data import Subset
    import random

    # Define subset sizes
    MAX_TRAIN_SAMPLES = 8000
    MAX_VAL_SAMPLES = 1000
    MAX_TEST_SAMPLES = 1000

    # Create subsets
    train_indices = random.sample(
        range(len(train_loader.dataset)),
        min(MAX_TRAIN_SAMPLES, len(train_loader.dataset))
    )
    val_indices = random.sample(
        range(len(val_loader.dataset)),
        min(MAX_VAL_SAMPLES, len(val_loader.dataset))
    )
    test_indices = random.sample(
        range(len(test_loader.dataset)),
        min(MAX_TEST_SAMPLES, len(test_loader.dataset))
    )

    # Create subset datasets
    train_subset = Subset(train_loader.dataset, train_indices)
    val_subset = Subset(val_loader.dataset, val_indices)
    test_subset = Subset(test_loader.dataset, test_indices)

    # Create new dataloaders
    train_loader = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=False
    )

    val_loader = DataLoader(
        val_subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=False
    )

    test_loader = DataLoader(
        test_subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=False
    )

    print(f"Using reduced subsets for testing:")
    print(f"  - Training: {len(train_loader.dataset)} samples")
    print(f"  - Validation: {len(val_loader.dataset)} samples")
    print(f"  - Test: {len(test_loader.dataset)} samples")

In [ ]:
# ============================================================================
# 6. SETUP TRAINING ENVIRONMENT
# ============================================================================

print("\n" + "="*70)
print("SETTING UP TRAINING ENVIRONMENT")
print("="*70)

# Import training functions
try:
    from train_classifier import (
        load_training_config,
        setup_environment,
        build_model,
        build_loss_function,
        build_optimizer,
        build_scheduler,
        train_one_epoch,
        validate_one_epoch,
        save_checkpoint,
        train_classifier,
        load_best_model
    )
    print("✓ Training functions imported successfully")
except ImportError as e:
    print(f"Error importing training functions: {e}")
    sys.exit(1)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Load complete configuration
full_config = load_training_config(
    training_cfg_path=str(config_dir / 'training.yaml'),
    model_cfg_path=str(config_dir / 'model.yaml')
)

# Create results directory structure
results_dir = Path(project_path) / 'results'
checkpoints_dir = results_dir / 'checkpoints'
comparison_dir = results_dir / 'model_comparison'

# Create directories
for directory in [results_dir, checkpoints_dir, comparison_dir]:
    directory.mkdir(parents=True, exist_ok=True)

# Update config with checkpoint directory
full_config['checkpoint_dir'] = str(checkpoints_dir)

print(f"\nResults directories created:")
print(f"  - {checkpoints_dir.relative_to(project_path)}")
print(f"  - {comparison_dir.relative_to(project_path)}")



SETTING UP TRAINING ENVIRONMENT
✓ Training functions imported successfully
Training device: cuda
GPU: Tesla T4
GPU Memory: 15.83 GB

Results directories created:
  - results/checkpoints
  - results/model_comparison


In [ ]:
# ============================================================================
# 7. TRAIN ALL MODELS
# ============================================================================

print("\n" + "="*70)
print("MODEL TRAINING PIPELINE")
print("="*70)

# Dictionaries to store results
all_models = {}
all_results = {}
all_predictions = {}

# Training parameters
num_epochs = training_config['training']['epochs']
early_stopping_patience = training_config['training']['early_stopping_patience']

print(f"\nTraining Configuration:")
print(f"  - Maximum epochs: {num_epochs}")
print(f"  - Early stopping patience: {early_stopping_patience}")
print(f"  - Mixed precision: {training_config['training']['mixed_precision']}")
print(f"  - Models to train: {MODELS_TO_TRAIN}")

# Train each model sequentially
for model_idx, MODEL_NAME in enumerate(MODELS_TO_TRAIN, 1):
    print(f"\n" + "="*70)
    print(f"TRAINING [{model_idx}/{len(MODELS_TO_TRAIN)}]: {MODEL_NAME.upper()}")
    print("="*70)

    # Verify model exists in configuration
    if MODEL_NAME not in model_config['models']:
        print(f"Model '{MODEL_NAME}' not found in configuration")
        continue

    # Display model configuration
    model_spec = model_config['models'][MODEL_NAME]
    print(f"\nModel Configuration:")
    print(f"  - Number of classes: {model_spec['num_classes']}")
    print(f"  - Pretrained: {model_spec['pretrained']}")
    print(f"  - Input size: {model_spec['input_size']}")
    print(f"  - Dropout: {model_spec['classifier']['dropout']}")
    print(f"  - Freeze backbone: {model_spec['fine_tuning']['freeze_backbone']}")

    try:
        print(f"\nStarting training process...")

        # Train the model
        training_result = train_classifier(
            config=full_config,
            model_name=MODEL_NAME,
            train_loader=train_loader,
            val_loader=val_loader,
            checkpoints_dir=checkpoints_dir
        )

        # Store results
        model = training_result['model']
        all_models[MODEL_NAME] = model
        all_results[MODEL_NAME] = {
            'history': training_result['history'],
            'best_val_acc': training_result['best_val_acc'],
            'best_epoch': training_result['best_epoch'],
            'model_config': model_spec
        }

        print(f"\n Training completed successfully!")
        print(f"  - Best validation accuracy: {training_result['best_val_acc']:.4f}")
        print(f"  - Best epoch: {training_result['best_epoch']}")
        print(f"  - Total epochs trained: {len(training_result['history']['train_loss'])}")

        # Save complete model
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        model_save_path = checkpoints_dir / f'{MODEL_NAME}_{timestamp}.pth'

        torch.save({
            'model_state_dict': model.state_dict(),
            'model_name': MODEL_NAME,
            'best_val_acc': training_result['best_val_acc'],
            'best_epoch': training_result['best_epoch'],
            'history': training_result['history'],
            'input_size': model_spec['input_size'],
            'model_config': model_spec,
            'training_config': training_config,
            'class_mapping': class_config
        }, model_save_path)

        print(f"  - Model saved: {model_save_path.relative_to(project_path)}")

        # Calculate parameter statistics
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  - Total parameters: {total_params:,}")
        print(f"  - Trainable parameters: {trainable_params:,}")

    except Exception as e:
        print(f"\n Error training {MODEL_NAME}: {e}")
        import traceback
        traceback.print_exc()
        continue

print("\n" + "="*70)
print("TRAINING PHASE COMPLETED")
print("="*70)

if all_models:
    print(f" Successfully trained {len(all_models)} model(s): {list(all_models.keys())}")
else:
    print(" No models were successfully trained")
    sys.exit(1)


Les details du training s'affichent ici


In [ ]:
# ============================================================================
# 8. GENERATE PREDICTIONS
# ============================================================================

print("\n" + "="*70)
print("GENERATING PREDICTIONS")
print("="*70)

# Prepare class names for evaluation
class_names = []
for i in range(10):  # Assuming 10 classes
    if str(i) in class_config['classes']:
        class_names.append(class_config['classes'][str(i)]['name'])
    else:
        class_names.append(f"Class_{i}")

# Dictionaries for predictions
test_predictions = {}
test_true_labels = []

print(f"\nGenerating predictions on test set...")

# Generate predictions for each trained model
for model_name, model in all_models.items():
    print(f"\n  {model_name.upper()}:")

    model_preds = []
    temp_test_labels = []

    model.eval()
    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(test_loader):
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            model_preds.extend(preds.cpu().numpy())
            temp_test_labels.extend(labels.numpy())

    # Store predictions
    test_predictions[model_name] = np.array(model_preds)

    # Store true labels (first model only)
    if model_name == list(all_models.keys())[0]:
        test_true_labels = np.array(temp_test_labels)

    # Calculate accuracy
    test_accuracy = np.mean(test_predictions[model_name] == test_true_labels)
    print(f"    • Test Accuracy: {test_accuracy:.4f}")

print(f"\n✓ Predictions generated for {len(all_models)} model(s)")
print(f"  - Test samples: {len(test_true_labels)}")


In [ ]:
# ============================================================================
# 9. SIMPLIFIED PREDICTION VISUALIZATIONS
# ============================================================================

print("\n" + "="*70)
print("PREDICTION VISUALIZATIONS FOR BOTH MODELS")
print("="*70)

# Create directory for visualizations
predictions_viz_dir = results_dir / 'prediction_visualizations'
predictions_viz_dir.mkdir(exist_ok=True)

print(f"Visualizations will be saved in: {predictions_viz_dir.relative_to(project_path)}")

# Function to denormalize image
def denormalize_image(tensor_image):
    """Denormalize image for visualization"""
    if isinstance(tensor_image, torch.Tensor):
        tensor_image = tensor_image.cpu().numpy()

    mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
    std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)
    tensor_image = tensor_image * std + mean
    tensor_image = np.clip(tensor_image, 0, 1)
    tensor_image = np.transpose(tensor_image, (1, 2, 0))
    return tensor_image

# Get a batch of test images (10 images)
test_batch = next(iter(test_loader))
test_images, test_labels = test_batch
test_images = test_images[:20]
test_labels = test_labels[:20]

print(f"\nGenerating visualizations for 8 test images...")

# Create visualization for each model
for model_name, model in all_models.items():
    print(f"\nProcessing {model_name.upper()}...")

    # Make predictions
    model.eval()
    with torch.no_grad():
        images_gpu = test_images.to(device)
        outputs = model(images_gpu)
        _, predictions = torch.max(outputs, 1)

    # Calculate batch accuracy
    batch_accuracy = (predictions.cpu() == test_labels).float().mean().item()

    # Create figure with 2x4 grid (10 images)
    fig, axes = plt.subplots(4, 5, figsize=(16, 8))
    axes = axes.flatten()

    for idx in range(20):
        ax = axes[idx]

        # Display image
        img = denormalize_image(test_images[idx])
        ax.imshow(img)

        # Get labels
        true_label = test_labels[idx].item()
        pred_label = predictions[idx].item()

        true_class = class_names[true_label]
        pred_class = class_names[pred_label]

        # Determine if prediction is correct
        is_correct = true_label == pred_label
        color = 'green' if is_correct else 'red'

        # Create title
        title = f"True: {true_class}\nPred: {pred_class}"
        ax.set_title(title, color=color, fontsize=11, pad=8)
        ax.axis('off')

    # Add overall title
    plt.suptitle(f'{model_name.upper()} - Test Predictions\n'
                 f'Batch Accuracy: {batch_accuracy:.1%}',
                 fontsize=14, fontweight='bold', y=1.02)

    plt.tight_layout()

    # Save figure
    viz_path = predictions_viz_dir / f'test_predictions_{model_name}.png'
    plt.savefig(viz_path, dpi=150, bbox_inches='tight')
    plt.show()

    print(f"  ✓ Saved: test_predictions_{model_name}.png")
    print(f"    Batch accuracy: {batch_accuracy:.1%}")

print(f"\n Visualizations completed!")
print(f"   Files saved in: {predictions_viz_dir.relative_to(project_path)}")

In [ ]:
# ============================================================================
# 10. MODEL EVALUATION AND COMPARISON
# ============================================================================

print("\n" + "="*70)
print("MODEL EVALUATION AND COMPARISON")
print("="*70)

# Import evaluation module
try:
    import evaluate
    print("✓ Evaluation module imported successfully")
except ImportError as e:
    print(f"Error importing evaluation module: {e}")
    # Skip evaluation if module not available
    evaluate = None

if evaluate and 'test_predictions' in locals() and test_predictions:
    # Prepare accuracy metrics
    train_acc = {}
    val_acc = {}
    test_acc = {}

    print(f"\nPreparing metrics for {len(all_models)} model(s)...")

    for model_name in all_models.keys():
        # Training accuracy (last epoch)
        if model_name in all_results:
            history = all_results[model_name]['history']
            if history['train_acc']:
                train_acc[model_name] = history['train_acc'][-1]
            else:
                train_acc[model_name] = 0.0

        # Validation accuracy (best)
        if model_name in all_results:
            val_acc[model_name] = all_results[model_name]['best_val_acc']

        # Test accuracy
        if model_name in test_predictions:
            test_acc[model_name] = np.mean(test_predictions[model_name] == test_true_labels)
        else:
            test_acc[model_name] = 0.0

    # Run comprehensive evaluation
    print(f"\nRunning model comparison...")

    try:
        evaluation_results = evaluate.evaluate_models(
            y_true=test_true_labels,
            predictions_dict=test_predictions,
            class_names=class_names,
            train_acc=train_acc,
            val_acc=val_acc,
            test_acc=test_acc,
            save_results=True,
            results_dir=str(comparison_dir)
        )

        # Identify best model
        best_model_name = evaluate.get_best_model(evaluation_results['metrics'])

        print(f"\n BEST MODEL: {best_model_name.upper()}")

        # Display comparison table
        print("\n MODEL PERFORMANCE COMPARISON:")
        print("-" * 60)
        print(f"{'Model':<20} {'Train':<8} {'Val':<8} {'Test':<8} {'F1-Score':<8}")
        print("-" * 60)

        for model_name, metrics in evaluation_results['metrics'].items():
            train = train_acc.get(model_name, 0)
            val = val_acc.get(model_name, 0)
            test = metrics['accuracy']
            f1 = metrics['f1']

            print(f"{model_name:<20} {train:.4f}  {val:.4f}  {test:.4f}  {f1:.4f}")

        print("-" * 60)

    except Exception as e:
        print(f"Error during evaluation: {e}")
        import traceback
        traceback.print_exc()
else:
    print("\n  Skipping detailed evaluation:")
    if not evaluate:
        print("  - Evaluation module not available")
    elif 'test_predictions' not in locals():
        print("  - No test predictions available")
    elif not test_predictions:
        print("  - Test predictions dictionary is empty")

# Display simple comparison if evaluation module not available
if 'test_predictions' in locals() and test_predictions and len(test_predictions) > 0:
    print(f"\n SIMPLE ACCURACY COMPARISON:")
    for model_name, preds in test_predictions.items():
        if len(preds) > 0 and len(test_true_labels) > 0:
            accuracy = np.mean(preds == test_true_labels)
            print(f"  {model_name.upper():20} → Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

In [ ]:
# ============================================================================
# 11. SAVE FINAL RESULTS
# ============================================================================

print("\n" + "="*70)
print("SAVING FINAL RESULTS")
print("="*70)

try:
    # Check if visualization directory exists
    predictions_viz_dir = results_dir / 'prediction_visualizations'
    has_visualizations = predictions_viz_dir.exists()

    # Get test accuracies if available
    test_acc = {}
    if 'test_predictions' in locals() and test_predictions:
        for model_name, preds in test_predictions.items():
            if len(preds) > 0 and len(test_true_labels) > 0:
                test_acc[model_name] = np.mean(preds == test_true_labels)

    # Get best model name if available
    best_model_name = None
    if test_acc:
        best_model_name = max(test_acc, key=test_acc.get)

    # Create comprehensive summary
    final_summary = {
        'project_info': {
            'project_name': 'Driver Distraction Detection',
            'training_date': datetime.now().strftime("%Y-%m-%d"),
            'training_time': datetime.now().strftime("%H:%M:%S"),
            'test_mode': TEST_MODE,
            'total_models_trained': len(all_models),
            'trained_models': list(all_models.keys()),
            'best_model': best_model_name,
            'device_used': str(device)
        },
        'dataset_info': {
            'training_samples': len(train_loader.dataset),
            'validation_samples': len(val_loader.dataset),
            'test_samples': len(test_loader.dataset),
            'total_samples': len(train_loader.dataset) + len(val_loader.dataset) + len(test_loader.dataset),
            'input_size': DEFAULT_INPUT_SIZE,
            'batch_size': batch_size,
            'num_classes': len(class_names)
        },
        'training_configuration': {
            'epochs': num_epochs,
            'early_stopping_patience': early_stopping_patience,
            'learning_rate': training_config['optimizer']['lr'],
            'optimizer': training_config['optimizer']['name'],
            'loss_function': training_config['loss']['name']
        },
        'visualization_info': {
            'prediction_visualizations_created': has_visualizations,
            'visualization_directory': str(predictions_viz_dir.relative_to(project_path)) if has_visualizations else None
        },
        'model_performance': {}
    }

    # Add performance for each model
    for model_name, results in all_results.items():
        final_summary['model_performance'][model_name] = {
            'best_epoch': results['best_epoch'],
            'best_validation_accuracy': float(results['best_val_acc']),
            'final_training_accuracy': float(results['history']['train_acc'][-1]) if results['history']['train_acc'] else 0.0,
            'final_validation_accuracy': float(results['history']['val_acc'][-1]) if results['history']['val_acc'] else 0.0,
            'test_accuracy': float(test_acc.get(model_name, 0)) if test_acc else 0.0
        }

    # Save summary
    summary_file = results_dir / 'training_summary.json'
    with open(summary_file, 'w', encoding='utf-8') as f:
        json.dump(final_summary, f, indent=2, default=str)

    print(f"✓ Final summary saved: {summary_file.relative_to(project_path)}")

    # Display results structure
    print(f"\n RESULTS STRUCTURE:")
    print(f"  results/")
    print(f"  ├── checkpoints/                    # Model weights")
    print(f"  │   ├── [model_name]_*.pth         # Timestamped models")
    print(f"  │   ├── [model_name]_best.pth      # Best validation model")
    print(f"  │   └── [model_name]_last.pth      # Last epoch model")
    print(f"  ├── model_comparison/              # Comparative analysis")
    print(f"  │   ├── visualizations/            # Comparison charts")
    print(f"  │   └── metrics/                   # Detailed metrics")
    print(f"  ├── prediction_visualizations/     # Test image predictions")
    print(f"  │   └── test_predictions_*.png     # Prediction samples")
    print(f"  └── training_summary.json          # Complete training summary")

except Exception as e:
    print(f"  Error saving final summary: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*70)
print(" TRAINING PIPELINE COMPLETED SUCCESSFULLY!")
print("="*70)
print(f" All results saved in: results/")
print(f" Models: results/checkpoints/")
if len(all_models) > 1:
    print(f" Comparison: results/model_comparison/")
print(f"  Visualizations: results/prediction_visualizations/")
print(f" Summary: results/training_summary.json")